In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from imblearn.over_sampling import RandomOverSampler


import warnings
warnings.filterwarnings('ignore')

In [4]:
df=pd.read_csv('notebook/data.csv')
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [5]:
df.shape

(381109, 12)

In [6]:
df['Response'].value_counts()

Response
0    334399
1     46710
Name: count, dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


**Data Preprocessing**

In [8]:
num_col = df.select_dtypes(include=['float64','int64']).columns
cat_col = df.select_dtypes(include = 'object').columns

In [9]:
df['Gender'] = df['Gender'].map({'Female':0,'Male':1}).astype(int)

In [10]:
# Categorical features and one hot encoding
df = pd.get_dummies(df,drop_first = True).astype(int)
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Vehicle_Age_< 1 Year,Vehicle_Age_> 2 Years,Vehicle_Damage_Yes
0,1,1,44,1,28,0,40454,26,217,1,0,1,1
1,2,1,76,1,3,0,33536,26,183,0,0,0,0
2,3,1,47,1,28,0,38294,26,27,1,0,1,1
3,4,1,21,1,11,1,28619,152,203,0,1,0,0
4,5,0,29,1,41,1,27496,152,39,0,1,0,0


In [11]:
num_feat = ['Age','Vintage']
cat_feat = ['Driving_License','Gender','Previously_Insured','Vehicle_Age_gt_2_Years','Vehicle_Age_lt_1_Year','Vehicle_Damage_Yes','Region_Code','Policy_Sales_Channel']

In [12]:
df = df.rename(columns = {'Vehicle_Age_< 1 Year':'Vehicle_Age_lt_1_Year','Vehicle_Age_> 2 Years':'Vehicle_Age_gt_2_Years'})
df['Vehicle_Age_lt_1_Year'] = df['Vehicle_Age_lt_1_Year'].astype('int')
df['Vehicle_Age_gt_2_Years'] = df['Vehicle_Age_gt_2_Years'].astype('int')
df['Vehicle_Damage_Yes'] = df['Vehicle_Damage_Yes'].astype('int')

for column in cat_feat:
    df[column] = df[column].astype('str')

In [13]:
# Scaling Data
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler

ss = StandardScaler()
mm = MinMaxScaler()

df[num_feat] = ss.fit_transform(df[num_feat])
df['Annual_Premium'] = df['Annual_Premium']

df.drop('id', axis=1, inplace=True)

In [14]:
# train,test splitting
from sklearn.model_selection import train_test_split

train_target = df['Response']
train = df.drop(['Response'],axis=1)

x_train,x_test,y_train,y_test = train_test_split(train,train_target,random_state=0)

### Model Training

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

random_search = {
    'criterion': ['entropy','gini'],
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 2,3,4,5,6,7,8,9,10],
    'min_samples_split': [2, 5,7,10],
    'min_samples_leaf': [4,6,8],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

clf = RandomForestClassifier()
model = RandomizedSearchCV(estimator=clf,param_distributions=random_search,n_iter=10,cv= 4,verbose = 1,random_state=101,n_jobs= -1)

model.fit(x_train,y_train)

Fitting 4 folds for each of 10 candidates, totalling 40 fits


,estimator,RandomForestClassifier()
,param_distributions,"{'bootstrap': [True, False], 'criterion': ['entropy', 'gini'], 'max_depth': [None, 2, ...], 'max_features': ['sqrt', 'log2'], ...}"
,n_iter,10
,scoring,None
,n_jobs,-1
,refit,True
,cv,4
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,101
,error_score,nan


In [16]:
best_params = model.best_params_
best_params

{'n_estimators': 300,
 'min_samples_split': 10,
 'min_samples_leaf': 8,
 'max_features': 'log2',
 'max_depth': 2,
 'criterion': 'gini',
 'bootstrap': True}

In [17]:
# Saving Model
import pickle

file_name = 'rf_model.pkl'
pickle.dump(model,open(file_name,'wb'))

### Model Evaluation

In [18]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.88      1.00      0.93     83603
           1       0.00      0.00      0.00     11675

    accuracy                           0.88     95278
   macro avg       0.44      0.50      0.47     95278
weighted avg       0.77      0.88      0.82     95278

